In [ ]:
%run ./utils_common

In [ ]:
logger = setup_logger("TotalPipelineSpendsReporter")

In [ ]:
dbutils.widgets.text("catalog", "", "CATALOG")
dbutils.widgets.text("schema", "", "SCHEMA")
dbutils.widgets.text("overlap_days", "3", "Overlap days (min 2)")

In [ ]:
# =======================================================
# Workload-type mapping (plan §3.1 / §5.5)
# =======================================================
# Friendly label per billing_origin_product. Unknown / new products fall back
# to the raw value downstream (coalesce) so nothing is ever dropped.
WORKLOAD_MAP = {
    "DLT": "DLT Pipeline",
    "SQL": "DBSQL Materialized View",
    "DATABASE": "Online Table",
    "VECTOR_SEARCH": "Vector Search",
    "MODEL_SERVING": "Model Serving",
    "AI_FUNCTIONS": "AI Functions",
}

# Single source of truth for "which workloads are expected to carry a
# system.lakeflow.pipelines snapshot" (plan §5.5). Vector Search / Model
# Serving / AI Functions never get a row, so they are excluded by design.
# Defined here as the canonical set; the backend §5.3 metadata-missing KPI
# mirrors it so the two never drift.
METADATA_BEARING_WORKLOADS = {
    "DLT Pipeline",
    "DBSQL Materialized View",
    "Online Table",
}

In [ ]:
# =======================================================
# Total Pipeline Spends Client
# =======================================================
# Sibling of TotalPoolSpendsClient. Rolls the per-cluster
# dbspend360_pipeline_dbu_cost staging table up into the denormalized
# dbspend360_total_pipeline_spends rollup. Differences vs the pool rollup:
#   * source DBU table: dbspend360_pipeline_dbu_cost
#                       (keyed (workspace_id, pipeline_id, usage_date,
#                        cluster_id, billing_origin_product))
#   * target table:     dbspend360_total_pipeline_spends
#                       (keyed (workspace_id, pipeline_id, usage_date,
#                        billing_origin_product) - cluster_id is aggregated
#                        away into compute_mode; billing_origin_product STAYS
#                        in the grain so the per-workload $ split is exact and
#                        reconciles row-for-row with staging - NO within-day
#                        dominant-product approximation. See plan §3.3 / §5.5).
#   * derived dimensions (the rollup is the ONLY place these are derived):
#       - workload_type : friendly label from billing_origin_product via
#                         WORKLOAD_MAP, raw value for unknowns (never dropped).
#       - compute_mode  : serverless / classic / mixed. A single
#                         (pipeline, day, product) can straddle serverless +
#                         classic clusters -> 'mixed'.
#       - cost_basis    : full (serverless) / dbu_only (classic) /
#                         partial (mixed). Drives the per-row UI honesty icon.
#   * metadata denorm:  SCD-collapse system.lakeflow.pipelines on
#                       (workspace_id, pipeline_id) and denormalize
#                       pipeline_name / pipeline_type / created_by / run_as /
#                       delete_time -> pipeline_deleted_at. created_by/run_as
#                       come straight from the system table (99.94% populated
#                       for DLT, plan §0/§3.4) - NO REST API, NO metadata cache
#                       (the key simplification vs the Instance Pools rollup).
#   * metadata_missing: computed BEFORE the COALESCE fallback on pipeline_name
#                       so it captures the underlying snapshot state, not the
#                       post-fallback state. Three-state, product-aware badge
#                       (plan §3.5): active / deleted-visible / metadata-not-
#                       available (the EXPECTED state for Vector Search etc.).
#   * NO cloud-cost join in v1 (only the ~4% classic spend has a separate
#     cloud-VM line). cloud_cost = CAST(NULL AS DOUBLE), reserved; staging
#     keeps cluster_id so the v2 cluster_id-keyed join needs no re-ingest.
#     total_cost = databricks_cost + COALESCE(cloud_cost, 0) so v2 picks up a
#     populated cloud_cost with no code change.
#   * MERGE key: (workspace_id, pipeline_id, usage_date,
#                 billing_origin_product) - all NON-nullable, so plain '=' is
#                 safe here. The null-safe concern is staging-only (§5.4): the
#                 nullable cluster_id is not in the rollup key.
class TotalPipelineSpendsClient:

    TABLE_NAME = "dbspend360_total_pipeline_spends"

    def __init__(
        self,
        audit_table: str,
        databricks_cost_table: str,
        target_table: str,
        error_log_table: str,
        overlap_days: int,
        logger=None,
    ):
        self.audit_table = audit_table
        self.databricks_cost_table = databricks_cost_table
        self.target_table = target_table
        self.error_log_table = error_log_table
        self.overlap_days = overlap_days
        self.logger = logger or logging.getLogger("TotalPipelineSpendsClient")

    def _load_pipeline_snapshot(self):
        # SCD-collapse system.lakeflow.pipelines to one row per
        # (workspace_id, pipeline_id) carrying the most-recent snapshot.
        # QUALIFY ROW_NUMBER() OVER (... ORDER BY change_time DESC) = 1 is
        # holistically safe on tied change_time (one winner per partition).
        # delete_time is non-null iff the pipeline was deleted; carry it
        # through as pipeline_deleted_at for the §3.5 "Deleted YYYY-MM-DD"
        # badge. workspace_id is in the partition because pipeline_id is only
        # unique within a workspace (plan §3.3).
        # The join keys are aliased to p_* so they never collide with the
        # day-grain columns. On serverless Spark Connect a shared-name column
        # that participates in an equi-join key becomes unresolvable when
        # referenced (qualified) after the join, so we keep every post-join
        # column uniquely named and reference only bare names downstream.
        return spark.sql("""
            SELECT workspace_id AS p_workspace_id,
                   pipeline_id  AS p_pipeline_id,
                   name AS pipeline_name,
                   pipeline_type,
                   created_by,
                   run_as,
                   delete_time AS pipeline_deleted_at
            FROM system.lakeflow.pipelines
            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY workspace_id, pipeline_id
                ORDER BY change_time DESC) = 1
        """)

    def build_total_pipeline_spends(self):
        start_dt = end_dt = datetime.now(timezone.utc).date()
        try:
            start_dt, end_dt = get_date_window(self.audit_table, self.TABLE_NAME, self.overlap_days)

            valid, msg = validate_date_window(start_dt, end_dt)
            if not valid:
                raise DataQualityError(msg)

            self.logger.info(
                f"Building dbspend360_total_pipeline_spends for {start_dt} → {end_dt}"
            )

            staging_df = (
                spark.table(self.databricks_cost_table)
                    .alias("stg")
                    .filter(
                        (F.col("usage_date") >= F.lit(start_dt)) &
                        (F.col("usage_date") <= F.lit(end_dt))
                    )
            )

            if staging_df.limit(1).count() == 0:
                self.logger.info(
                    "No pipeline DBU rows in this date window; nothing to roll up."
                )
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt,
                    "SUCCESS", 0, "No DBU data in window; v1: cloud cost not yet computed",
                )
                return

            # 1) Collapse staging (per cluster) to pipeline-day-PRODUCT.
            #    billing_origin_product STAYS in the grain (§3.3) so the
            #    per-workload $ split is exact - NO dominant-product collapse.
            #    A single (pipeline, day, product) can still straddle
            #    serverless + classic clusters -> 'mixed' compute_mode.
            day_df = (
                staging_df.groupBy(
                    "workspace_id", "pipeline_id", "usage_date", "billing_origin_product"
                )
                .agg(
                    F.sum("databricks_cost").alias("databricks_cost"),
                    F.sum("update_cost").alias("update_cost"),
                    F.sum("maintenance_cost").alias("maintenance_cost"),
                    F.when(F.countDistinct("compute_mode") > 1, F.lit("mixed"))
                     .otherwise(F.first("compute_mode")).alias("compute_mode"),
                    F.concat_ws(" + ", F.array_sort(F.collect_set("sku_name"))).alias("sku_name"),
                    F.first("currency").alias("currency"),
                )
                .withColumn(
                    "cost_basis",
                    F.when(F.col("compute_mode") == "serverless", F.lit("full"))
                     .when(F.col("compute_mode") == "classic", F.lit("dbu_only"))
                     .otherwise(F.lit("partial")),
                )
            )

            # workload_type from billing_origin_product; unknowns fall back to
            # the raw value (coalesce) so new/unmapped products are never lost.
            mapping = F.create_map([F.lit(x) for kv in WORKLOAD_MAP.items() for x in kv])
            day_df = day_df.withColumn(
                "workload_type",
                F.coalesce(mapping[F.col("billing_origin_product")], F.col("billing_origin_product")),
            )

            # 2) SCD-collapse system.lakeflow.pipelines and LEFT-join metadata.
            #    LEFT so pipeline-days with no snapshot row still land in the
            #    rollup; metadata_missing is the signal (set BEFORE the COALESCE
            #    fallback paints a synthetic pipeline_name).
            pipelines_df = self._load_pipeline_snapshot()

            # Join keys live only on the day-grain side (p_* on the snapshot
            # side), so every post-join column is a unique bare name.
            joined = (
                day_df
                .join(
                    pipelines_df,
                    on=(
                        (F.col("workspace_id") == F.col("p_workspace_id")) &
                        (F.col("pipeline_id") == F.col("p_pipeline_id"))
                    ),
                    how="left",
                )
                .withColumn("metadata_missing", F.col("pipeline_name").isNull())
            )

            select_cols = [
                F.col("workspace_id"),
                F.col("pipeline_id"),
                F.col("usage_date"),
                F.coalesce(
                    F.col("pipeline_name"),
                    F.concat(F.lit("Pipeline "), F.col("pipeline_id")),
                ).alias("pipeline_name"),
                F.col("pipeline_type"),
                F.col("created_by"),
                F.col("run_as"),
                F.col("workload_type"),
                F.col("compute_mode"),
                F.col("cost_basis"),
                F.col("metadata_missing"),
                F.col("pipeline_deleted_at"),
                F.col("databricks_cost"),
                F.col("update_cost"),
                F.col("maintenance_cost"),
                # --------------------------------------------------------
                # TODO(v2): classic pipeline cloud cost join goes here.
                # When dbspend360_cloud_cost_explorer (cluster_id-tagged)
                # is wired in, LEFT-join it on cluster_id (kept in staging,
                # so NO re-ingest) and project the resulting cloud_cost in
                # place of the F.lit(None) below. The total_cost expression
                # already picks up a populated cloud_cost without any change.
                # --------------------------------------------------------
                F.lit(None).cast("double").alias("cloud_cost"),
                F.col("currency"),
                F.col("sku_name"),
                F.col("billing_origin_product"),
            ]

            final_df = joined.select(*select_cols)

            final_df = (
                final_df
                .withColumn(
                    "total_cost",
                    F.coalesce(F.col("databricks_cost"), F.lit(0.0))
                    + F.coalesce(F.col("cloud_cost"), F.lit(0.0)),
                )
                .withColumn("created_at", F.current_timestamp())
                .withColumn("updated_at", F.current_timestamp())
            )
            final_df = safe_cache(final_df)

            row_count = final_df.count()

            validate_source_schema(
                final_df,
                {"workspace_id": "string", "pipeline_id": "string",
                 "usage_date": "date", "billing_origin_product": "string",
                 "workload_type": "string", "compute_mode": "string",
                 "cost_basis": "string", "metadata_missing": "boolean",
                 "databricks_cost": "double", "cloud_cost": "double",
                 "total_cost": "double"},
                self.target_table, self.logger,
            )
            validate_no_negative_costs(
                final_df,
                ["databricks_cost", "update_cost", "maintenance_cost",
                 "cloud_cost", "total_cost"],
                self.target_table, self.logger,
            )
            validate_currency_consistency(final_df, "currency", self.target_table, self.logger)

            target = DeltaTable.forName(spark, self.target_table)
            (target.alias("t")
                .merge(
                    final_df.alias("s"),
                    # All four key columns are NON-nullable, so plain '=' is
                    # safe (the null-safe cluster_id concern is staging-only,
                    # §5.4 - cluster_id is aggregated out of the rollup key).
                    "t.workspace_id = s.workspace_id "
                    "AND t.pipeline_id = s.pipeline_id "
                    "AND t.usage_date = s.usage_date "
                    "AND t.billing_origin_product = s.billing_origin_product",
                )
                .whenMatchedUpdate(set={
                    "pipeline_name": "s.pipeline_name",
                    "pipeline_type": "s.pipeline_type",
                    "created_by": "s.created_by",
                    "run_as": "s.run_as",
                    "workload_type": "s.workload_type",
                    "compute_mode": "s.compute_mode",
                    "cost_basis": "s.cost_basis",
                    "metadata_missing": "s.metadata_missing",
                    "pipeline_deleted_at": "s.pipeline_deleted_at",
                    "databricks_cost": "s.databricks_cost",
                    "update_cost": "s.update_cost",
                    "maintenance_cost": "s.maintenance_cost",
                    "cloud_cost": "s.cloud_cost",
                    "total_cost": "s.total_cost",
                    "currency": "s.currency",
                    "sku_name": "s.sku_name",
                    "updated_at": "current_timestamp()",
                })
                .whenNotMatchedInsert(values={
                    "workspace_id": "s.workspace_id",
                    "pipeline_id": "s.pipeline_id",
                    "usage_date": "s.usage_date",
                    "pipeline_name": "s.pipeline_name",
                    "pipeline_type": "s.pipeline_type",
                    "created_by": "s.created_by",
                    "run_as": "s.run_as",
                    "workload_type": "s.workload_type",
                    "compute_mode": "s.compute_mode",
                    "cost_basis": "s.cost_basis",
                    "metadata_missing": "s.metadata_missing",
                    "pipeline_deleted_at": "s.pipeline_deleted_at",
                    "databricks_cost": "s.databricks_cost",
                    "update_cost": "s.update_cost",
                    "maintenance_cost": "s.maintenance_cost",
                    "cloud_cost": "s.cloud_cost",
                    "total_cost": "s.total_cost",
                    "currency": "s.currency",
                    "sku_name": "s.sku_name",
                    "billing_origin_product": "s.billing_origin_product",
                    "created_at": "current_timestamp()",
                    "updated_at": "current_timestamp()",
                })
                .execute()
            )

            safe_unpersist(final_df)
            get_merge_metrics(self.target_table, self.logger)

            validate_post_merge(
                self.target_table, "usage_date",
                start_dt, end_dt, row_count, self.logger,
            )

            # No reconciliation invariant in v1 (cloud_cost is always NULL;
            # total_cost == databricks_cost). Stamp the audit message so the
            # absence is visible and gets dropped in v2 when cloud_cost lands.
            log_audit_run(
                self.audit_table, self.TABLE_NAME, start_dt, end_dt,
                "SUCCESS", row_count, "v1: cloud cost not yet computed",
            )
            self.logger.info(
                f"Merged {row_count} rows into {self.target_table} "
                f"for {start_dt} → {end_dt}."
            )

        except Exception as e:
            msg = str(e)[:1000]
            self.logger.error(f"Run failed: {msg}")
            try:
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt, "FAILED", 0, msg,
                )
            except Exception:
                self.logger.error("Failed to write FAILED audit entry")
            raise

In [ ]:
# =======================================================
# APP
# =======================================================
class TotalPipelineSpendsApp:

    def __init__(self):
        catalog = dbutils.widgets.get("catalog")
        schema = dbutils.widgets.get("schema")
        ov_days = get_overlap_days(dbutils.widgets.get("overlap_days"), logger=logger)

        self.client = TotalPipelineSpendsClient(
            audit_table=build_table_fqn(catalog, schema, "dbspend360_audit_log"),
            databricks_cost_table=build_table_fqn(catalog, schema, "dbspend360_pipeline_dbu_cost"),
            target_table=build_table_fqn(catalog, schema, "dbspend360_total_pipeline_spends"),
            error_log_table=build_table_fqn(catalog, schema, "dbspend360_error_log"),
            overlap_days=ov_days,
            logger=logger,
        )

    def run(self):
        self.client.build_total_pipeline_spends()

In [ ]:
# =======================================================
# Execute
# =======================================================
app = TotalPipelineSpendsApp()
app.run()